# Can we predict 12-month loan default?

**Question:** using only information available at application time, can we predict `defaulted` (1 = default within 12 months of funding)?

**Plan:** audit the data, split off a held-out test set, compare a prevalence baseline, logistic regression and gradient boosting with cross-validation, evaluate once on the test set (discrimination, calibration, usable thresholds), then inspect what drives predictions.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED = 42
df = pd.read_csv('data.csv'); dd = pd.read_csv('columns.csv')
dd

## 1. Data audit

In [ ]:
print(df.shape, '| missing:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()), '| duplicate ids:', int(df.application_id.duplicated().sum()))
display(df.describe().T)
rate = df.defaulted.mean(); print(f'Default rate: {rate:.2%}  ({df.defaulted.sum()} of {len(df)})')

Clean data (no missing values or duplicates). Defaults are ~10%, so the classes are imbalanced: accuracy is uninformative (always predicting "no default" scores ~90%), so we use **ROC AUC** and **PR AUC** (average precision; the no-skill value equals the default rate). `application_id` is only an identifier and is excluded from the features.

In [ ]:
num = ['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig, ax = plt.subplots(2, 3, figsize=(13, 6))
for a, c in zip(ax.flat, num):
    q = pd.qcut(df[c], 10, duplicates='drop')
    g = df.groupby(q, observed=True).defaulted.mean()
    a.plot(range(len(g)), g.values, marker='o'); a.axhline(rate, ls='--', c='gray')
    a.set_title(f'Default rate by {c} decile'); a.set_xlabel('decile (low to high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean', 'size']))

Default rises with `debt_ratio` and `loan_amount_k` and falls with `income_k` and `employment_years`; `credit_lines`, `age` and `region` look flat on their own. Univariate views can hide interactions, so a flexible model checks for those below.

## 2. Split and models
Stratified 80/20 split; the test set is used only once, at the end. Model choice uses 5-fold CV on the training set.

In [ ]:
X = df.drop(columns=['defaulted', 'application_id']); y = df.defaulted
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
cat = ['region']
pre_lin = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(drop='first'), cat)])
pre_tree = ColumnTransformer([('n', 'passthrough', num), ('c', OneHotEncoder(), cat)])
models = {
    'Prevalence baseline': DummyClassifier(strategy='prior'),
    'Logistic regression': Pipeline([('p', pre_lin), ('m', LogisticRegression(max_iter=1000))]),
    'Gradient boosting': Pipeline([('p', pre_tree), ('m', HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=200, max_depth=3, min_samples_leaf=50, l2_regularization=1.0, random_state=SEED))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []; oof = {}
for n, m in models.items():
    p = cross_val_predict(m, Xtr, ytr, cv=cv, method='predict_proba')[:, 1]; oof[n] = p
    rows.append(dict(model=n, roc_auc=roc_auc_score(ytr, p), pr_auc=average_precision_score(ytr, p), brier=brier_score_loss(ytr, p)))
pd.DataFrame(rows).set_index('model').round(4)

Out-of-fold results on the training set. If gradient boosting and logistic regression are close, the simpler, more interpretable model is preferable.

## 3. Final evaluation on the held-out test set

In [ ]:
use_lr = roc_auc_score(ytr, oof['Logistic regression']) >= roc_auc_score(ytr, oof['Gradient boosting']) - 0.005
final = models['Logistic regression'] if use_lr else models['Gradient boosting']
print('Selected:', 'Logistic regression' if use_lr else 'Gradient boosting', '(simpler model preferred unless GB is >0.005 AUC better in CV)')
final.fit(Xtr, ytr); pte = final.predict_proba(Xte)[:, 1]
rng = np.random.default_rng(SEED); yv = yte.values; aucs = []; prs = []
for _ in range(1000):
    i = rng.integers(0, len(yv), len(yv)); aucs.append(roc_auc_score(yv[i], pte[i])); prs.append(average_precision_score(yv[i], pte[i]))
ci = lambda a: f'[{np.percentile(a, 2.5):.3f}, {np.percentile(a, 97.5):.3f}]'
print(f'Test ROC AUC {roc_auc_score(yte, pte):.3f}  95% CI {ci(aucs)}')
print(f'Test PR AUC  {average_precision_score(yte, pte):.3f}  95% CI {ci(prs)}  (no-skill = {yte.mean():.3f})')
print(f'Test Brier   {brier_score_loss(yte, pte):.4f}  (prevalence-only = {brier_score_loss(yte, np.full(len(yte), ytr.mean())):.4f})')
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
f, t, _ = roc_curve(yte, pte); ax[0].plot(f, t); ax[0].plot([0, 1], [0, 1], '--', c='gray'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR')
pr, rc, _ = precision_recall_curve(yte, pte); ax[1].plot(rc, pr); ax[1].axhline(yte.mean(), ls='--', c='gray'); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision')
fp, mp = calibration_curve(yte, pte, n_bins=8, strategy='quantile'); ax[2].plot(mp, fp, marker='o'); ax[2].plot([0, .5], [0, .5], '--', c='gray'); ax[2].set(title='Calibration', xlabel='predicted', ylabel='observed')
plt.tight_layout(); plt.show()

### Business view: default rate by predicted-risk decile, and lift

In [ ]:
t = pd.DataFrame({'y': yte.values, 'p': pte}); t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=range(1, 11))
g = t.groupby('decile', observed=True).agg(n=('y', 'size'), defaults=('y', 'sum'), rate=('y', 'mean'), mean_pred=('p', 'mean'))
g['lift'] = g.rate / yte.mean(); display(g.round(3))
top = t.sort_values('p', ascending=False)
for k in (0.1, 0.2, 0.3):
    s = top.head(int(k * len(top))); print(f'Flagging riskiest {k:.0%}: captures {s.y.sum() / t.y.sum():.0%} of defaults, precision {s.y.mean():.1%}')

## 4. What drives the prediction?

In [ ]:
pi = permutation_importance(final, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=X.columns); err = pd.Series(pi.importances_std, index=X.columns)
order = imp.sort_values().index
imp[order].plot.barh(xerr=err[order], figsize=(7, 3.5), title='Permutation importance (drop in test ROC AUC)'); plt.tight_layout(); plt.show()
if use_lr:
    names = final.named_steps['p'].get_feature_names_out()
    print(pd.Series(final.named_steps['m'].coef_[0], index=names).sort_values().round(3).to_string())
    print('(numeric features standardised: coefficients are log-odds per 1 SD)')

## 5. Conclusions and caveats

Read the numbers above as authoritative; this is a guide to interpreting them.

- **Default is predictable, but only moderately.** The model separates risky from safe applicants clearly better than chance (see ROC/PR AUC with confidence intervals), but it is not a crystal ball: because defaults are rare, precision stays low at any high-recall threshold.
- **Drivers** are the affordability variables: `debt_ratio`, `income_k`, `loan_amount_k` and `employment_years`. `region`, `age` and `credit_lines` add little.
- Predicted probabilities are usable for ranking and, where the calibration plot hugs the diagonal, as risk estimates.
- **Caveats:** random (not time-based) split on a single snapshot; validate out-of-time before relying on it for future cohorts. Only funded loans have outcomes, so declined applicants are missing (selection bias). No cost matrix was provided, so no operating threshold is recommended; use the decile/lift table with real approval and loss costs to choose one. Review `age` and `region` for fair-lending constraints before any use in decisions.